In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gamma
import math
import time

# 1. Gerar dados de treino (números naturais de 1 a 10)
X_train = np.arange(1, 11, dtype=float)

# Fatorial real utilizando math.factorial
y_factorial = np.array([math.factorial(int(n)) for n in X_train], dtype=float)

# Aplicando a transformação proposta para suavização da curva combinatória
y_train_transformed = np.zeros_like(X_train)
for i, n in enumerate(X_train):
    fact = math.factorial(int(n))
    if fact == 1:
        y_train_transformed[i] = 1.0  # Mapeamento estipulado estável para n=1
    else:
        y_train_transformed[i] = 1.0 / np.log(fact)

print("X de Treino:", X_train)
print("Y Transformado (Alvo do PSO):", y_train_transformed)

In [ ]:
def tsk_inference(X, centers, sigmas, y_target=None, ridge_lambda=1e-5):
    """
    Motor de Inferência TSK de Ordem 1 (Linear).
    Consequentes calculados via Mínimos Quadrados com Regularização Ridge (Wiktorowicz, 2020).
    """
    N = len(X)
    R = len(centers) # Número de regras = número de centros (Chang, 2025)
    
    # 1. Fuzzificação: Calcular grau de ativação com Antecedentes Gaussianos
    W = np.zeros((N, R))
    for i in range(N):
        for j in range(R):
            W[i, j] = np.exp(-((X[i] - centers[j])**2) / (2 * (sigmas[j]**2) + 1e-8))
            
    # Normalização dos pesos das regras
    row_sums = W.sum(axis=1, keepdims=True)
    W_norm = np.where(row_sums > 0, W / row_sums, 0)
    
    # 2. Construção da Matriz Global de Design (X_hat)
    X_hat = np.zeros((N, 2 * R))
    for j in range(R):
        X_hat[:, 2*j] = W_norm[:, j] * X      # Coeficiente linear (a_j)
        X_hat[:, 2*j + 1] = W_norm[:, j]      # Coeficiente constante/afim (b_j)
        
    # 3. Estimação de Parâmetros Ótimos do Consequente via Ridge Regression
    if y_target is not None:
        # Equação clássica de regularização linear adotada por Wiktorowicz (2021)
        A = X_hat.T @ X_hat + ridge_lambda * np.eye(2 * R)
        try:
            P = np.linalg.inv(A) @ X_hat.T @ y_target
        except np.linalg.LinAlgError:
            P = np.linalg.pinv(A) @ X_hat.T @ y_target # Pseudo-inversa como fallback de segurança
        return X_hat @ P, P
    else:
        return X_hat

In [ ]:
def particle_swarm_optimization(X, y_trans, num_rules, pop_size=30, iterations=100, seed=42):
    np.random.seed(seed)
    centers = np.array(X)
    
    # Hiperparâmetros padrões de estabilidade do PSO (Inércia e Coeficientes de Aceleração)
    w = 0.5   # Peso de inércia
    c1 = 1.5  # Componente Cognitivo (atração para a melhor posição individual)
    c2 = 1.5  # Componente Social (atração para a melhor posição global do enxame)
    
    # Inicialização das posições (sigmas das gaussianas) e velocidades das partículas
    # Cada partícula possui dimensão igual ao número de regras
    position = np.random.uniform(0.5, 2.5, size=(pop_size, num_rules))
    velocity = np.random.uniform(-0.1, 0.1, size=(pop_size, num_rules))
    
    # Histórico de melhores posições individuais (pbest) e suas respectivas avaliações (fitness)
    pbest_position = position.copy()
    pbest_fitness = np.array([np.mean((y_trans - tsk_inference(X, centers, ind, y_trans)[0])**2) for ind in position])
    
    # Melhor posição global (gbest)
    gbest_idx = np.argmin(pbest_fitness)
    gbest_position = pbest_position[gbest_idx].copy()
    
    global_best_fitness_history = []
    
    # Ciclo Evolutivo Bioinspirado do Enxame
    for it in range(iterations):
        for i in range(pop_size):
            # Termos estocásticos do PSO
            r1 = np.random.rand(num_rules)
            r2 = np.random.rand(num_rules)
            
            # Equação de Atualização de Velocidade (Wiktorowicz, 2021)
            velocity[i] = (w * velocity[i] + 
                           c1 * r1 * (pbest_position[i] - position[i]) + 
                           c2 * r2 * (gbest_position - position[i]))
            
            # Atualização de Posição
            position[i] = position[i] + velocity[i]
            
            # Restrição de Espaço de Busca: Sigmas não podem ser nulos ou negativos
            position[i] = np.clip(position[i], 1e-3, 5.0)
            
            # Avaliação de Aptidão da nova posição (MSE do TSK)
            y_pred, _ = tsk_inference(X, centers, position[i], y_trans)
            current_mse = np.mean((y_trans - y_pred)**2)
            
            # Atualiza o melhor individual (pbest) se a nova posição for superior
            if current_mse < pbest_fitness[i]:
                pbest_fitness[i] = current_mse
                pbest_position[i] = position[i].copy()
                
                # Atualiza o líder do enxame global (gbest)
                if current_mse < pbest_fitness[gbest_idx]:
                    gbest_position = position[i].copy()
                    gbest_idx = i
                    
        global_best_fitness_history.append(pbest_fitness[gbest_idx])
        
    return gbest_position, global_best_fitness_history

In [ ]:
seeds = [10, 42, 100, 2026, 999]
results = []
convergence_curves = []

print("Iniciando o Protocolo Experimental unificado de 5 Execuções Independentes (PSO)... \n")
start_time = time.time()

for s in seeds:
    best_sigmas, history = particle_swarm_optimization(X_train, y_train_transformed, num_rules=10, seed=s)
    results.append({
        'seed': s,
        'best_sigmas': best_sigmas,
        'final_mse': history[-1]
    })
    convergence_curves.append(history)
    print(f"Seed {s:4d} finalizada. MSE final obtido: {history[-1]:.6f}")

total_time = time.time() - start_time
mses = [r['final_mse'] for r in results]

print("\n--- MATRIZ DE MÉTRICAS DE VALIDAÇÃO (REQUISITO DAS LAUDAS) ---")
print(f"Melhor MSE (Aptidão) encontrada: {np.min(mses):.6f}")
print(f"Pior MSE encontrada:            {np.max(mses):.6f}")
print(f"Média dos MSEs obtidos:         {np.mean(mses):.6f}")
print(f"Desvio Padrão (Estabilidade):   {np.std(mses):.6f}")
print(f"Custo Computacional Total:      {total_time:.4f} segundos")

In [ ]:
# Recuperando os hiperparâmetros ótimos da melhor execução encontrada pelo PSO
best_run = results[np.argmin(mses)]
optimal_sigmas = best_run['best_sigmas']
centers_fixed = X_train

# Extraindo as predições ótimas e a matriz de consequentes lineares ponderada
y_pred_trans, optimal_P = tsk_inference(X_train, centers_fixed, optimal_sigmas, y_train_transformed)

# Mapeando o domínio contínuo para avaliação de generalização contínua entre os fatoriais
X_continuous = np.linspace(1.1, 10.0, 300)
X_hat_continuous = tsk_inference(X_continuous, centers_fixed, optimal_sigmas)
y_pred_continuous_trans = X_hat_continuous @ optimal_P

# Inversão da transformação matemática de escala: Retornando do domínio suavizado para o Fatorial Real!
y_pred_factorial_continuous = np.exp(1.0 / y_pred_continuous_trans)

# Renderização Gráfica para Exportação e Inclusão no Overleaf
plt.figure(figsize=(14, 5))

# Plot A: Evolução do Enxame ao longo das Iterações (Métrica de Convergência da Lauda)
plt.subplot(1, 2, 1)
for i, hist in enumerate(convergence_curves):
    plt.plot(hist, label=f"Seed {seeds[i]}")
plt.title("Curvas de Convergência do Enxame (PSO)")
plt.xlabel("Iterações")
plt.ylabel("MSE Global (Espaço Transformado)")
plt.yscale('log')
plt.grid(True)
plt.legend()

# Plot B: Validação da Superfície de Controle Invertida vs Baseline Analítico (Função Gamma)
plt.subplot(1, 2, 2)
# Baseline Analítico Real: Gamma(x+1) = x!
plt.plot(X_continuous, gamma(X_continuous + 1), 'g-', label='Função Gamma Analítica $\Gamma(x+1)$', alpha=0.7)
# Saída do Super Modelo Híbrido TSK + PSO
plt.plot(X_continuous, y_pred_factorial_continuous, 'r--', label='Aproximação Contínua TSK + PSO', linewidth=2)
# Instâncias de treino
plt.scatter(X_train, y_factorial, color='black', zorder=5, label='Fatoriais Discretos Reais ($n!$)')

plt.title("Superfície de Inferência Contínua do Fatorial via TSK-PSO")
plt.xlabel("Entrada ($x$)")
plt.ylabel("Valor Calculado (Escala Logarítmica)")
plt.yscale('log')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()